# AMR Scout — pilot: carbapenemase genes in *Klebsiella pneumoniae*

Reproducible analysis of isolates returned by an **NCBI Pathogen Detection
`collection_date:2023*` query** (Isolates Browser TSV export, `isolates (1).tsv`).

**Scope note.** The export does **not** contain a collection-date column, so the dataset is
described only as "isolates returned by a `collection_date:2023*` query" and **no temporal
analysis is performed**. The `Create date` column is a record timestamp, not a sampling date.

**Genotype is not phenotype.** Every count below is a gene *detection* in an assembly
(AMRFinderPlus call). It is not a measured MIC and does not establish clinical resistance.
Carbapenem resistance also arises without a carbapenemase (e.g. porin loss), and some
carbapenemase variants have reduced carbapenem activity (see "Functional reclassification").

**Sampling.** These genomes are a convenience sample that laboratories chose to sequence and
deposit, enriched for outbreak and resistance investigations. Percentages describe the sample,
not population prevalence.

### Requirements
`numpy`, `pandas`, network access to `ftp.ncbi.nlm.nih.gov`. Figure rendering uses `matplotlib`
if importable, otherwise the bundled dependency-free renderer `prender.py` (`fonttools` only).

## 1. Load & clean

In [ ]:
import pandas as pd, numpy as np, re, collections, urllib.request, io, os

ISOLATES_TSV = "isolates (1).tsv"   # NCBI Pathogen Detection Isolates Browser export

# keep_default_na=False: empty cells stay as "" so placeholder detection is explicit,
# and gene strings are never coerced to NaN.
df = pd.read_csv(ISOLATES_TSV, sep="\t", dtype=str, keep_default_na=False, na_values=[])
print("rows before cleaning:", len(df), "| columns:", len(df.columns))
df.head(5)

In [ ]:
# --- Location cleaning -------------------------------------------------------
# Drop empty, NULL, "missing..." and other null-in-disguise placeholders.
PLACEHOLDER = re.compile(
    r"^(null|not collected|not applicable|not determined|not provided|unknown|n/?a|none|-|\?+|missing.*)$",
    re.I)

loc  = df["Location"].astype(str).str.strip()
drop = loc.eq("") | loc.str.contains("missing", case=False) | loc.str.match(PLACEHOLDER)

print("dropped:", int(drop.sum()))
print(df.loc[drop, "Location"].replace("", "<empty>").value_counts().to_dict())

clean = df.loc[~drop].copy()
# country = text before the first colon, e.g. "China: Xinjiang" -> "China"
clean["country"] = clean["Location"].str.split(":", n=1).str[0].str.strip()
print("rows after cleaning:", len(clean), "| distinct countries:", clean["country"].nunique())
clean["country"].value_counts().head(15)

In [ ]:
# Country-level comparisons are restricted to countries with >= 30 isolates.
MIN_ISOLATES = 30
cc = clean["country"].value_counts()
keep_countries = cc[cc >= MIN_ISOLATES].index
print(f"{len(keep_countries)} countries with n>={MIN_ISOLATES}, covering {int(cc[cc>=MIN_ISOLATES].sum())} isolates")

# Data-quality flags worth carrying into any downstream join:
#  * labels are NCBI-style, not ISO-3166 ("USA", "Viet Nam", "South Korea", "Czech Republic")
#  * "Hong Kong", "Taiwan", "Guadeloupe", "Gaza Strip" are territories, not sovereign states
#  * counts measure sequencing/submission activity, not national burden

## 2. Gene flagging

In [ ]:
def genes(s):
    """AMR genotypes cell -> list of gene symbols, suffixes (=COMPLETE/=PARTIAL/=POINT) stripped."""
    out = []
    for tok in str(s).split(","):
        tok = tok.strip()
        if tok:
            out.append(tok.split("=", 1)[0].strip())
    return out

clean["_genes"] = clean["AMR genotypes"].map(genes)
all_genes = collections.Counter(g for gl in clean["_genes"] for g in gl)
print("distinct gene symbols:", len(all_genes),
      "| isolates with empty genotype:", int(clean["_genes"].map(len).eq(0).sum()))

In [ ]:
# --- NCBI AMRFinderPlus Reference Gene Catalog: authoritative subclass per allele ---
CATALOG_URL = ("https://ftp.ncbi.nlm.nih.gov/pathogen/Antimicrobial_resistance/"
               "AMRFinderPlus/database/latest/ReferenceGeneCatalog.txt")
raw = urllib.request.urlopen(CATALOG_URL, timeout=120).read().decode("utf-8", "replace")
rg  = pd.read_csv(io.StringIO(raw), sep="\t", dtype=str, low_memory=False)
print("catalog rows:", len(rg), "| db_version:", rg["db_version"].dropna().iloc[0])

rg["key_allele"] = rg["allele"].fillna("").str.lower()
rg["key_fam"]    = rg["gene_family"].fillna("").str.lower()
allele_sub = dict(zip(rg.key_allele, rg.subclass.fillna("")))
fam_sub    = {}
for k, v in zip(rg.key_fam, rg.subclass.fillna("")):
    fam_sub.setdefault(k, v)      # first (family-level) entry wins

def subclass_of(g):
    """Catalog subclass for a gene symbol; allele match first, gene_family as fallback."""
    k = g.lower()
    if allele_sub.get(k): return allele_sub[k].upper()
    if fam_sub.get(k):    return fam_sub[k].upper()
    return None

In [ ]:
# OXA-48-like is an explicit allele list (never a blaOXA prefix: that would sweep in
# blaOXA-1 / blaOXA-9 / blaOXA-10, which are not carbapenemases).
# blaOXA-484 added: NCBI classes it as an OXA-48-family CARBAPENEM enzyme.
# blaOXA-1201 deliberately excluded: OXA-48 family by sequence, but subclass CEPHALOSPORIN.
OXA48 = {"blaoxa-48", "blaoxa-181", "blaoxa-232", "blaoxa-244",
         "blaoxa-162", "blaoxa-204", "blaoxa-484"}

# Naive (family-prefix) flags, kept for comparison with the functional rule in section 3.
def fam_naive(gl):
    low = [g.lower() for g in gl]
    return dict(
        KPC         = any(g.startswith("blakpc") for g in low),
        NDM         = any(g.startswith("blandm") for g in low),
        OXA_48_like = any(g in OXA48 for g in low),
        VIM         = any(g.startswith("blavim") for g in low),
        IMP         = any(g.startswith("blaimp") for g in low),
    )
naive = pd.DataFrame([fam_naive(gl) for gl in clean["_genes"]], index=clean.index)
print((naive.sum() / len(naive) * 100).round(2).to_string())

In [ ]:
# Audit: every OXA-containing bla gene NOT on the OXA-48-like list, with its catalog subclass.
oxa_other = {g: n for g, n in all_genes.items()
             if "oxa" in g.lower() and g.lower().startswith("bla") and g.lower() not in OXA48}
pd.DataFrame(
    [(g, n, subclass_of(g),
      (rg.loc[rg.key_allele == g.lower(), "product_name"].iloc[0]
       if (rg.key_allele == g.lower()).any() else ""))
     for g, n in sorted(oxa_other.items(), key=lambda x: -x[1])],
    columns=["gene", "n_isolates", "subclass", "product_name"])

## 3. Functional reclassification

A gene is counted only when its catalog subclass set contains `CARBAPENEM`.

Two details that materially change the numbers:

1. **Compound subclass labels are split on `/`.** `blaNDM-9` is `CARBAPENEM/TANIBORBACTAM` and
   `blaKPC-160` is `CARBAPENEM/CEFIDEROCOL`; a string-equality test would silently drop them.
2. **24 `blaKPC` alleles carry no `CARBAPENEM` token** (e.g. `blaKPC-33`, `blaKPC-31`,
   `blaKPC-71`) — the ceftazidime-avibactam-associated variants whose carbapenemase activity is
   reduced (PMID 33558042, doi:10.1016/j.jiac.2021.01.014; PMID 33915286,
   doi:10.1016/j.cmi.2021.04.016). Both columns are therefore reported:
   `KPC_family_present` (any `blaKPC`) and `KPC_functional` (CARBAPENEM subclass).

`ompK35`/`ompK36` porin mutations carry subclass CARBAPENEM in the catalog but are **not**
carbapenemases and are excluded from every count.

In [ ]:
def is_carb(g):
    s = subclass_of(g)
    return s is not None and "CARBAPENEM" in s.split("/")   # compound labels count

def classify(g):
    """Gene symbol -> carbapenemase family, or None."""
    k = g.lower()
    if k.startswith("omp"):  return None     # porin mutations are not carbapenemases
    if not is_carb(g):       return None     # functional gate
    if k.startswith("blakpc"): return "KPC"
    if k.startswith("blandm"): return "NDM"
    if k in OXA48:             return "OXA_48_like"
    if k.startswith("blavim"): return "VIM"
    if k.startswith("blaimp"): return "IMP"
    return "Other"                            # GES-5, GES-24, OXA-72, AFM, L1, NWM-1, ...

FAMS = ["KPC", "NDM", "OXA_48_like", "VIM", "IMP", "Other"]
flags = pd.DataFrame([{f: f in ({classify(g) for g in gl} - {None}) for f in FAMS}
                      for gl in clean["_genes"]], index=clean.index)
flags["any_carbapenemase"] = flags[FAMS].any(axis=1)
flags["n_families"]        = flags[FAMS].sum(axis=1)

# both KPC definitions
kpc_family = clean["_genes"].map(lambda gl: any(g.lower().startswith("blakpc") for g in gl))
pa = pd.concat([clean[["Isolate", "BioSample", "Assembly", "Location", "country"]],
                kpc_family.rename("KPC_family_present"),
                flags.rename(columns={"KPC": "KPC_functional"})], axis=1)
pa.to_csv("carbapenemase_presence_absence.tsv", sep="\t", index=False)

N = len(pa)
cols = ["KPC_family_present", "KPC_functional", "NDM", "OXA_48_like",
        "VIM", "IMP", "Other", "any_carbapenemase"]
prev = pd.DataFrame({"n": [int(pa[c].sum()) for c in cols]}, index=cols)
prev["pct"] = (prev.n / N * 100).round(2)
prev.index.name = "family"
prev.to_csv("carbapenemase_prevalence.csv")
print("N =", N)
print(prev.to_string())
print("\nKPC family present but not functional:", int((kpc_family & ~flags["KPC"]).sum()))
print(">=2 families:", int((flags.n_families >= 2).sum()),
      f"({(flags.n_families >= 2).mean()*100:.2f}%)")

In [ ]:
# Sanity check: every catalog-CARBAPENEM gene present in the data is either assigned
# to a family or is an excluded porin. Nothing should fall through silently.
present_carb = {g for g in all_genes if is_carb(g)}
unassigned   = {g for g in present_carb if classify(g) is None and not g.lower().startswith("omp")}
assert not unassigned, unassigned
print("porins excluded:", sorted({g for g in present_carb if g.lower().startswith("omp")})[:6], "...")
print("'Other' family genes:", sorted({g for g in all_genes if classify(g) == "Other"}))

## 4. Figure

`matplotlib` is used when it imports. On hosts where it does not (this analysis was produced on
a Windows machine whose Application Control policy blocks `ft2font`, `_backend_agg`, `_path`,
`kiwisolver` and Pillow's `_imaging`), the bundled `prender.py` renders the same figure with
numpy + fontTools only and writes a true 300-dpi PNG.

In [ ]:
pct = (pa[pa.country.isin(keep_countries)]
       .groupby("country")[["KPC_functional","NDM","OXA_48_like","VIM","IMP","Other","any_carbapenemase"]]
       .mean() * 100)
pct["n"] = pa[pa.country.isin(keep_countries)].groupby("country").size()
pct = pct.loc[pa.country.value_counts().head(10).index]          # 10 largest by isolate count
pct = pct.sort_values("any_carbapenemase", ascending=False)      # sorted by % any carbapenemase
pct.round(2)

In [ ]:
TITLE = ("Carbapenemase genes in K. pneumoniae genomes "
         "(NCBI Pathogen Detection, collection_date:2023* query)")
YLAB  = "% of isolates carrying gene family"
OUT   = "carbapenemase_by_country.png"
# Okabe-Ito colorblind-safe palette; the union bar is neutral black so it does not
# read as a seventh gene family.
SERIES = [("Any carbapenemase (union)", "any_carbapenemase", (0.12, 0.12, 0.12)),
          ("KPC",         "KPC_functional", (0.000, 0.447, 0.698)),
          ("NDM",         "NDM",            (0.835, 0.369, 0.000)),
          ("OXA-48-like", "OXA_48_like",    (0.000, 0.620, 0.451)),
          ("VIM",         "VIM",            (0.800, 0.475, 0.655)),
          ("IMP",         "IMP",            (0.337, 0.706, 0.914)),
          ("Other",       "Other",          (0.620, 0.620, 0.620))]
YMAX = 90.0

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as e:
    HAVE_MPL = False
    print("matplotlib unavailable -> prender fallback:", type(e).__name__, e)
print("HAVE_MPL =", HAVE_MPL)

In [ ]:
if HAVE_MPL:
    import numpy as np
    fig, ax = plt.subplots(figsize=(9.2, 5.0))
    x = np.arange(len(pct)); w = 0.86 / len(SERIES)
    for i, (lab, col, rgb) in enumerate(SERIES):
        ax.bar(x + i*w - 0.43 + w/2, pct[col].values, width=w*0.92, color=rgb, label=lab)
    ax.set_ylim(0, YMAX); ax.set_ylabel(YLAB); ax.set_title(TITLE, loc="left")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{c}\nn = {int(n):,}" for c, n in zip(pct.index, pct['n'])])
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, color="0.85", lw=0.6); ax.set_axisbelow(True)
    ax.legend(frameon=False, ncol=len(SERIES), loc="upper center",
              bbox_to_anchor=(0.5, 1.10), fontsize=7)
    fig.savefig(OUT, dpi=300, bbox_inches="tight")
    print("wrote", OUT, "via matplotlib")

In [ ]:
if not HAVE_MPL:
    # prender.py must sit next to this notebook; it needs only numpy + fontTools.
    import sys, os
    sys.path.insert(0, os.getcwd())
    import prender

    DPI, W_IN, H_IN = 300, 9.2, 5.0
    c = prender.Canvas(W_IN, H_IN, DPI); W, H = c.W, c.H
    L, R, T, B = 300, 60, 250, 300
    x0, x1, y0, y1 = L, W - R, T, H - B
    BASE, MID, TICK = 34, 29, 27
    INK, GREY, GRID = (0.10,)*3, (0.45,)*3, (0.85,)*3
    ypx = lambda v: y1 - (v / YMAX) * (y1 - y0)

    for v in range(0, int(YMAX) + 1, 10):
        c.line(x0, ypx(v), x1, ypx(v), GRID, 1.6)
        c.text(x0 - 16, ypx(v), f"{v}", TICK, GREY, ha="right", va="middle")
    c.line(x0, y1, x1, y1, INK, 2.2)
    c.text(70, (y0 + y1) / 2, YLAB, BASE, INK, ha="center", rot=90)

    slot = (x1 - x0) / len(pct); GW = slot * 0.86
    gap  = GW * 0.035; bw = (GW - gap) / len(SERIES)
    for gi, (country, row) in enumerate(pct.iterrows()):
        gx = x0 + slot * gi + (slot - GW) / 2
        for si, (lab, col, rgb) in enumerate(SERIES):
            bx = gx + si * bw + (gap if si else 0)
            hgt = y1 - ypx(float(row[col]))
            # §6.1: a zero-valued family gets a visible baseline stub, never an empty slot
            c.rect(bx, (y1 - 3) if hgt < 3 else ypx(float(row[col])),
                   bw * 0.92, 3 if hgt < 3 else hgt, rgb)
        cx = x0 + slot * gi + slot / 2
        c.text(cx, y1 + 46, country, MID, INK, ha="center", va="top")
        c.text(cx, y1 + 88, f"n = {int(row['n']):,}", TICK, GREY, ha="center", va="top")

    # title with K. pneumoniae italicised
    tx = x0
    for t, st in [("Carbapenemase genes in ", "regular"), ("K. pneumoniae", "italic"),
                  (" genomes (NCBI Pathogen Detection, collection_date:2023* query)", "regular")]:
        tx += c.text(tx, 78, t, BASE, INK, style=st)
    lx, ly = x0, 170
    for lab, col, rgb in SERIES:
        c.rect(lx, ly - 24, 30, 30, rgb); lx += 42
        lx += c.text(lx, ly, lab, MID, INK) + 40
    c.text(x0, H - 110, "Gene counted only when its NCBI Reference Gene Catalog subclass is "
                        "CARBAPENEM; ompK35/ompK36 porin mutations excluded.", 25, GREY)
    c.text(x0, H - 72,  "Bare 'blaOXA' calls left unclassified. Countries = 10 largest by isolate "
                        "count, each n>=30, sorted by % any carbapenemase. Genotype is not phenotype.",
           25, GREY)
    print("wrote", OUT, c.save_png(OUT), "via prender")

## Limitations

1. **Bare `blaOXA` calls (889 isolates, 6.15%)** have no resolved allele. The catalog maps the
   bare symbol to the OXA-1 node, so most are probably not carbapenemases, but they are left
   unclassified and every percentage here could shift slightly if they were resolved.
2. **No collection date**, so no temporal analysis; `Create date` is a record timestamp.
3. **Country labels are NCBI-style, not ISO-3166**, and include territories.
4. **Convenience sample.** Percentages describe sequenced-and-deposited genomes, enriched for
   outbreak and resistance investigations — not population prevalence
   (contrast population-level estimates: PMID 35065702, doi:10.1016/S0140-6736(21)02724-0).
5. **Genotype is not phenotype.** No MICs are available here; confirmation requires
   susceptibility testing (PMID 34234121, doi:10.1038/s41467-021-24448-3).